<a href="https://colab.research.google.com/github/dudinha-web/fundamentos-de-ia/blob/main/Aula_10_Lab_titanic_kaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Titanic Survival Prediction

---

## 1. Introdução
O naufrágio do Titanic (1912) é um dos eventos mais estudados em ciência de dados. Este notebook foi estruturado como **aula introdutória**, explicando **cada coluna**, **cada gráfico** e **cada etapa** do processo de Machine Learning.

---

## Objetivo
Prever a variável **Survived**:
- 0 → Não sobreviveu
- 1 → Sobreviveu

---

## 2. Dicionário de Dados (Explicação de TODAS as Colunas)

| Coluna | Descrição |
|------|----------|
| PassengerId | Identificador único do passageiro |
| Survived | Variável alvo (0 = morreu, 1 = sobreviveu) |
| Pclass | Classe do ticket (1 = alta, 2 = média, 3 = baixa) |
| Name | Nome do passageiro |
| Sex | Sexo |
| Age | Idade |
| SibSp | Nº de irmãos/cônjuges a bordo |
| Parch | Nº de pais/filhos a bordo |
| Ticket | Número do ticket |
| Fare | Valor pago pela passagem |
| Cabin | Cabine |
| Embarked | Porto de embarque (C, Q, S) |


Link KAGGLE - https://www.kaggle.com/competitions/titanic/overview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Carregamento dos dados

In [ ]:
df = pd.read_csv('/content/Titanic-Dataset.csv')
df.head()

In [ ]:
df.info()

# Análise Exploratória

In [ ]:
sns.countplot(x='Survived', data=df)
plt.show()

In [ ]:
sns.countplot(x='Sex', hue='Survived', data=df)
plt.show()

In [ ]:
sns.countplot(x='Pclass', hue='Survived', data=df)
plt.show()

In [ ]:
# Distribuição da variável alvo
sns.countplot(x='Survived', data=df)
plt.title('Distribuição de Sobreviventes')
plt.show()

In [ ]:
# Sobrevivência por Sexo
sns.countplot(x='Sex', hue='Survived', data=df)
plt.title('Sobrevivência por Sexo')
plt.show()

In [ ]:

# Classe Social
sns.countplot(x='Pclass', hue='Survived', data=df)
plt.title('Sobrevivência por Classe')
plt.show()

In [ ]:
# Distribuição de Idade
sns.histplot(df['Age'], kde=True)
plt.title('Distribuição de Idade')
plt.show()

In [ ]:

# Fare (Preço da passagem)
sns.histplot(df['Fare'], kde=True)
plt.title('Distribuição do Fare')
plt.show()


In [ ]:
# Correlação entre variáveis
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Correlação')
plt.show()

## Tratamento de Dados

In [ ]:
# Preenchimento

df['Age'].fillna(df['Age'].median(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

In [ ]:
# Remoção de colunas

df.drop(['Cabin', 'Name', 'Ticket'], axis=1, inplace=True)



In [ ]:
# Feature Engineering
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1


In [ ]:
# Encoding

df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)


In [ ]:
# Preparação dos Dados
X = df.drop('Survived', axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
# Normalização

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

In [ ]:
log_model = LogisticRegression(random_state=42, solver='liblinear', max_iter=1000)
log_model.fit(X_train, y_train)

log_pred = log_model.predict(X_test)
print('Logistic Regression:', accuracy_score(y_test, log_pred))
print('Random Forest:', accuracy_score(y_test, rf_pred))

In [ ]:
# Matriz de confusão

cm = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm, annot=True, fmt='d')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
#  Relatório

print(classification_report(y_test, rf_pred))


In [ ]:
# Validação Cruzada
scores = cross_val_score(rf_model, X, y, cv=5)
print('Cross-validation:', scores.mean())



In [ ]:

# Importância das Variáveis

rf_model.fit(X_train, y_train)
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances.sort_values().plot(kind='barh')
plt.title('Feature Importance')
plt.show()


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# Usa o DataFrame 'df' que foi carregado e pré-processado.
# 'Survived' é a variável alvo para o treinamento.
y_submission = df["Survived"]

# Define as features para este modelo, ajustando para o estado atual de 'df'
# (onde 'Sex' foi transformado em 'Sex_male').
features_for_model = ["Pclass", "Sex_male", "SibSp", "Parch"]

# Prepara X para treinamento usando as features selecionadas do 'df'.
X_submission = df[features_for_model]

# Obtém o 'PassengerId' para o arquivo de submissão.
# Como não há um conjunto de dados de teste separado carregado,
# vamos usar os 'PassengerId' do próprio 'df' para demonstração da geração do arquivo de submissão.
passenger_ids = df["PassengerId"]

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)
model.fit(X_submission, y_submission)

# Faz as previsões no mesmo conjunto de dados (X_submission) para demonstrar a criação do arquivo.
predictions = model.predict(X_submission)

# Cria o DataFrame de saída com 'PassengerId' e as previsões 'Survived'.
output = pd.DataFrame({'PassengerId': passenger_ids, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Seu arquivo para submeter esta disponivel !")